# 05. ハイパラ調整: 木の3モデルを強くしても、混ぜたスコアは伸びにくい

**このノートブックの目的:** LightGBM・XGBoost・CatBoostのハイパラ(木の大きさ・正則化・間引きの割合など)をOptunaで探し、単体と、ほかのモデルと混ぜたときのスコアがどれだけ伸びるかを確かめる。

**使うもの:** 03・04で作った36個の特徴量と、既定のハイパラのOOF。探索の記録(`data/optuna_lightgbm.db`・`optuna_xgboost.db`)があれば読み込む。無ければ、採用した値(`src/modeling.py` の `TUNED_LIGHTGBM_PARAMS`・`TUNED_XGBOOST_PARAMS`)で学習だけ行う。

**結論:** 調整で単体のOOFは伸びたが(LightGBM +0.00041)、RealMLPと混ぜると伸びは約1/9に縮んだ。調整した木はRealMLPに似た予測になり、伸びた分が重なったため。ここから、単体を強くするより仕組みの違うモデルを増やす方に時間を使うことにした。

## 準備

- ライブラリと、共通処理(`src/`の自分で書いたファイル)を読み込む。`src/features.py` はデータの読み込みと特徴量の作成、`src/modeling.py` はモデルの学習と評価。
- train・testを読み込み、目的変数 `satisfaction` を0/1の配列 `y` にする。
- Optunaを読み込み、採用した36個の特徴量の表(`X`・`X_test`)を作る。`RUN_SEARCH = False` のときは探索をやり直さず、保存した探索の記録を読む(Trueにすると1モデル50試行で数時間かかる)。

In [1]:
import sys

sys.path.insert(0, "../src")  # 共通処理(features.py・modeling.py・public_oof.py)を読み込む
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import features
import modeling

plt.rcParams["font.family"] = "Meiryo"  # グラフの日本語ラベル用
DATA = features.DATA_DIR
train, test = features.load_train_and_test()
y = train[features.TARGET].astype(int).values
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
X, X_test = features.build_train_and_test_tables(train, test)
RUN_SEARCH = False  # Trueにすると探索をやり直す(1モデル50試行で数時間かかる)

train 699,635行・test 299,844行、満足率 0.444


## 1. LightGBM

**目的:** 既定のハイパラ(葉31枚・正則化なし)のままでは、木が大きすぎたり小さすぎたりしている可能性がある。単体を強くすれば、RealMLPと混ぜたスコアも伸びるかを確かめる。

**検証方法:**
- Optuna(TPE)で8つのハイパラを同時に50試行探す: 葉の数・葉の最少人数・L2/L1正則化・分岐に必要な最小の改善・値の区切りの数・行の間引き・特徴量の間引き。1試行目は既定の設定。
- 評価は同じ5分割のOOF AUC。見込みの薄い試行はfoldの途中で打ち切る。
- 上位3つの設定を、fold分割のseedを変えた3通りで調整前と比べ、最良の設定(試行14)の学習率を0.05→0.02に下げて採用した(学習率を下げると、木の本数は増えるが少し伸びる)。

In [2]:
lightgbm_storage = f"sqlite:///{(DATA / 'optuna_lightgbm.db').as_posix()}"
if RUN_SEARCH:
    modeling.run_optuna_search("lightgbm", modeling.prepare_tuning_folds(X, y), y, 50, lightgbm_storage, "lightgbm")
if (DATA / "optuna_lightgbm.db").exists():
    study = optuna.load_study(study_name=optuna.get_all_study_names(lightgbm_storage)[0], storage=lightgbm_storage)
    trials = study.trials_dataframe().dropna(subset=["value"]).sort_values("value", ascending=False)
    display(trials[["number", "value"] + [c for c in trials.columns if c.startswith("params_")]].head(5).round(4))
print("採用した設定:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in modeling.TUNED_LIGHTGBM_PARAMS.items()})

,number,value,params_colsample_bytree,params_max_bin,params_min_child_samples,params_min_split_gain,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample
14,14,0.9608,0.5146,511,5,0.3485,61,0.3399,29.2660,0.9040
44,44,0.9608,0.4599,511,5,0.3173,61,3.4333,28.4723,0.9569
37,37,0.9608,0.4241,511,10,0.2859,122,1.3179,15.5525,0.9641
48,48,0.9608,0.5458,511,6,0.2870,69,1.4204,13.7618,0.9372
8,8,0.9608,0.6457,511,21,0.4436,35,0.3550,4.4466,0.8855


採用した設定: {'objective': 'binary', 'learning_rate': 0.02, 'n_estimators': 10000, 'random_state': 42, 'verbose': -1, 'num_leaves': 61, 'min_child_samples': 5, 'reg_lambda': 29.266, 'reg_alpha': 0.3399, 'min_split_gain': 0.3485, 'max_bin': 511, 'subsample': 0.904, 'subsample_freq': 1, 'colsample_bytree': 0.5146}


In [3]:
oof_lightgbm_tuned, test_lightgbm_tuned = modeling.cross_validate_lightgbm(X, y, params=modeling.TUNED_LIGHTGBM_PARAMS, X_test=X_test)
np.save(DATA / "oof_lightgbm_tuned.npy", oof_lightgbm_tuned)
np.save(DATA / "test_lightgbm_tuned.npy", test_lightgbm_tuned)
oof_lightgbm_default = np.load(DATA / "oof_lightgbm_default.npy")
print(f"LightGBM単体: 既定 {roc_auc_score(y, oof_lightgbm_default):.6f} → 調整後 {roc_auc_score(y, oof_lightgbm_tuned):.6f}")

fold0: AUC 0.960929 木1564本 (109秒)


fold1: AUC 0.959871 木1753本 (236秒)


fold2: AUC 0.961610 木1298本 (346秒)


fold3: AUC 0.961127 木1293本 (455秒)


fold4: AUC 0.960775 木1158本 (566秒)


LightGBM単体: 既定 0.960408 → 調整後 0.960853


### RealMLPと混ぜたときの伸び

**検証方法:** RealMLPとLightGBMの予測を順位に直し、重みを学習側の行で選んで平均する(5分割で、採点する行の答えは重みの選択に使わない)。調整前と調整後のLightGBMで、混ぜたOOF AUCを比べる。予測の似かたは順位相関で見る。

In [4]:
oof_realmlp = np.load(DATA / "oof_realmlp.npy")
rows = []
for label, oof_lightgbm in (("既定", oof_lightgbm_default), ("調整後", oof_lightgbm_tuned)):
    blended = modeling.cross_validate_rank_blend(np.column_stack([oof_lightgbm, oof_realmlp]), y)
    rows.append({"LightGBM": label, "LightGBM単体": roc_auc_score(y, oof_lightgbm), "RealMLPと混ぜたOOF": roc_auc_score(y, blended),
                 "RealMLPとの順位相関": modeling.rank_correlation_table({"a": oof_lightgbm, "b": oof_realmlp}).iloc[0, 1]})
comparison = pd.DataFrame(rows).set_index("LightGBM")
comparison.loc["調整による伸び"] = comparison.loc["調整後"] - comparison.loc["既定"]
comparison.round(6)

,LightGBM単体,RealMLPと混ぜたOOF,RealMLPとの順位相関
LightGBM,,,
既定,0.960408,0.961415,0.962389
調整後,0.960853,0.961472,0.969853
調整による伸び,0.000445,0.000057,0.007464


#### 結果と考察

- 調整でLightGBM単体は伸びたが、RealMLPと混ぜたOOFの伸びはずっと小さかった(記録では単体+0.00041に対し、混ぜると+0.000046で約1/9)。
- 調整後のLightGBMはRealMLPとの順位相関が上がった(記録では0.962→0.970)。調整で伸びた分は、RealMLPがすでに当てていた行の改善が多く、混ぜると重なったと考えられる。

## 2. XGBoost

**検証方法:** LightGBMと同じ流れで、深さ・葉の最少の重み・L2/L1正則化・分岐に必要な最小の改善・値の区切りの数・行と特徴量の間引きを50試行探し、最良の設定(試行35)を採用した。

In [5]:
xgboost_storage = f"sqlite:///{(DATA / 'optuna_xgboost.db').as_posix()}"
if RUN_SEARCH:
    modeling.run_optuna_search("xgboost", modeling.prepare_tuning_folds(X, y), y, 50, xgboost_storage, "xgboost")
if (DATA / "optuna_xgboost.db").exists():
    study = optuna.load_study(study_name=optuna.get_all_study_names(xgboost_storage)[0], storage=xgboost_storage)
    trials = study.trials_dataframe().dropna(subset=["value"]).sort_values("value", ascending=False)
    display(trials[["number", "value"] + [c for c in trials.columns if c.startswith("params_")]].head(5).round(4))
oof_xgboost_tuned, test_xgboost_tuned = modeling.cross_validate_xgboost(X, y, params=modeling.TUNED_XGBOOST_PARAMS, X_test=X_test)
np.save(DATA / "oof_xgboost_tuned.npy", oof_xgboost_tuned)
np.save(DATA / "test_xgboost_tuned.npy", test_xgboost_tuned)
print(f"XGBoost単体: 既定 {roc_auc_score(y, np.load(DATA / 'oof_xgboost_default.npy')):.6f} → 調整後 {roc_auc_score(y, oof_xgboost_tuned):.6f}")

,number,value,params_colsample_bytree,params_gamma,params_max_bin,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample
35,35,0.9608,0.4517,0.0981,1023,10,23.2775,0.1060,17.1544,0.9653
14,14,0.9608,0.4631,0.0772,1023,7,12.2669,0.0327,23.3404,0.8377
50,50,0.9607,0.4987,0.0032,1023,9,10.2702,0.0526,46.8350,0.9459
31,31,0.9607,0.4988,0.0792,1023,9,8.2710,0.0419,29.8362,0.9258
41,41,0.9607,0.4522,0.2063,1023,7,3.7836,0.0584,27.1069,0.9604


fold0: AUC 0.960827 木419本 (81秒)


fold1: AUC 0.959739 木454本 (177秒)


fold2: AUC 0.961555 木460本 (267秒)


fold3: AUC 0.961036 木549本 (370秒)


fold4: AUC 0.960576 木380本 (450秒)


XGBoost単体: 既定 0.960447 → 調整後 0.960741


## 3. CatBoost(KaggleのGPU)

**検証方法:** KaggleのGPUで、深さ・L2正則化・値の区切りの数・分岐の乱数の強さ・行の重みの乱数・カテゴリの組み合わせの最大数を30試行探した(`src/kaggle_gpu/tune_catboost.py`)。最良の設定(試行21)で学び直したOOFが `data/oof_catboost_tuned.npy`(04で読み込んだもの)。

**結果:** 単体は+0.000076伸びた。

## Summary

| モデル | 単体の伸び | 混ぜたときの伸び |
|---|---|---|
| LightGBM | +0.00041 | RealMLPと混ぜて+0.000046 |
| XGBoost | +0.000244 | 5モデルのスタックで+0.000005 |
| CatBoost | +0.000076 | 5モデルのスタックで+0.000020 |

(数値は実験記録の値。再実行すると、計算の順番の違いで小数第5〜6位が少しずれることがある)

ハイパラ調整の伸びは、混ぜると大きく縮んだ。次は06で、5つのモデルを重みを学んで混ぜる。